# Model Training - Random Forest with SMOTE
This notebook trains the predictive model on cleaned data.

In [ ]:
# Cell 1: Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE  # Using SMOTE as requested
import joblib

In [ ]:
# Cell 2: Load Cleaned Data
df = pd.read_csv('../data/processed/training_data_cleaned.csv')

In [ ]:
# Cell 3: Preprocessing (Encoding)
df = pd.get_dummies(df, columns=['Road_Type', 'Weather'], drop_first=True)

# Features (X) vs Target (y)
# We use Corrected_Severity as Target
X = df.drop(columns=['Record_ID', 'Reported_Severity', 'Corrected_Severity', 'NLP_Risk_Flag'])
y = df['Corrected_Severity']

In [ ]:
# Cell 4: Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Cell 5: Apply SMOTE (Synthetic Minority Over-sampling Technique)
print(f"Original Training Count: {len(X_train)}")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"SMOTE Resampled Count: {len(X_train_resampled)}")

In [ ]:
# Cell 6: Train Model
print("Training Random Forest on Resampled Data...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_resampled, y_train_resampled)
print("Training Complete.")

In [ ]:
# Cell 7: Evaluate
y_pred = rf.predict(X_test)

print("\nConfusion Matrix:")
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title("Actual vs Predicted High-Risk Accidents")
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Cell 8: Feature Importance
importances = rf.feature_importances_
features = X.columns
feat_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis')
plt.title('What Drives Accident Risk?')
plt.show()